# 06 - AI Model Comparison

Test and compare different AI models for invoice extraction:
- **Anthropic Claude** (claude-sonnet-4-5-20250929)
- **OpenAI GPT** (gpt-4o)
- **DeepSeek** (deepseek-chat)
- **Groq** (llama-3.2-90b-vision)
- **Ollama** (local models)

This notebook helps you find the best model for your use case.

In [ ]:
from src.ai_extractor import AIInvoiceExtractor
from src.document_processor import DocumentProcessor
from src.config import config, Config
from pathlib import Path
import json
from datetime import datetime
import time

In [ ]:
# Get test invoice
doc_processor = DocumentProcessor(config.directories.invoices)
files = doc_processor.list_invoice_files()

if not files:
    print("No invoice files found. Add some to data/invoices/")
else:
    test_file = files[0]  # Use first file for testing
    print(f"Test file: {test_file.name}")
    print(f"File type: {'PDF' if doc_processor.is_pdf(test_file) else 'Image'}")
    
    # Load file
    base64_data, media_type = doc_processor.file_to_base64(test_file)
    print(f"Media type: {media_type}")
    is_pdf = media_type == 'application/pdf'

## Provider Configuration

Configure which providers to test. Make sure you have API keys in your `.env` file.

**Note:** PDF extraction only works with Anthropic Claude (has native PDF support).

In [ ]:
# Configure providers to test
# Set to True/False based on which API keys you have

PROVIDERS_TO_TEST = {
    'anthropic': {
        'enabled': True,
        'models': ['claude-sonnet-4-5-20250929'],
        'supports_pdf': True
    },
    'openai': {
        'enabled': False,  # Set to True if you have OpenAI API key
        'models': ['gpt-4o', 'gpt-4o-mini'],
        'supports_pdf': False
    },
    'deepseek': {
        'enabled': False,  # Set to True if you have DeepSeek API key
        'models': ['deepseek-chat'],
        'supports_pdf': False
    },
    'groq': {
        'enabled': False,  # Set to True if you have Groq API key
        'models': ['llama-3.2-90b-vision-preview'],
        'supports_pdf': False
    },
    'ollama': {
        'enabled': False,  # Set to True if you have Ollama running locally
        'models': ['llama3.2-vision'],
        'supports_pdf': False
    }
}

print("Enabled providers:")
for provider, settings in PROVIDERS_TO_TEST.items():
    if settings['enabled']:
        print(f"  ✓ {provider}: {settings['models']}")

In [ ]:
# Test a single provider
def test_provider(provider_name, model_name):
    """Test extraction with a specific provider and model."""
    print(f"\n{'='*60}")
    print(f"Testing: {provider_name} / {model_name}")
    print(f"{'='*60}")
    
    try:
        # Create config for this provider
        test_config = Config.load_from_yaml()
        test_config.ai.provider = provider_name
        test_config.ai.model = model_name
        
        # Initialize extractor
        extractor = AIInvoiceExtractor(test_config)
        
        # Measure time
        start_time = time.time()
        
        # Extract data
        if is_pdf and not PROVIDERS_TO_TEST[provider_name]['supports_pdf']:
            print("⚠️  Skipped: This provider doesn't support PDF extraction")
            return None
        
        if is_pdf:
            invoice_data = extractor.extract_from_pdf(base64_data, source_file=test_file.name)
        else:
            invoice_data = extractor.extract_from_image(base64_data, media_type, source_file=test_file.name)
        
        elapsed = time.time() - start_time
        
        print(f"\n✓ Extraction successful ({elapsed:.2f}s)")
        print(f"\nExtracted data:")
        print(f"  Invoice #: {invoice_data.invoice_number}")
        print(f"  Date: {invoice_data.issue_date}")
        print(f"  Supplier: {invoice_data.supplier_name}")
        print(f"  Amount: {invoice_data.total_amount} {invoice_data.currency or 'CZK'}")
        
        if invoice_data.line_items:
            print(f"  Line items: {len(invoice_data.line_items)}")
        
        return {
            'provider': provider_name,
            'model': model_name,
            'time': elapsed,
            'success': True,
            'data': invoice_data.model_dump(),
            'error': None
        }
        
    except Exception as e:
        print(f"\n✗ Error: {e}")
        return {
            'provider': provider_name,
            'model': model_name,
            'time': 0,
            'success': False,
            'data': None,
            'error': str(e)
        }

In [ ]:
# Run tests for all enabled providers
results = []

if 'files' not in locals() or not files:
    print("No test file available. Run previous cells first.")
else:
    for provider, settings in PROVIDERS_TO_TEST.items():
        if not settings['enabled']:
            continue
        
        for model in settings['models']:
            result = test_provider(provider, model)
            if result:
                results.append(result)
    
    print(f"\n\n{'='*60}")
    print(f"COMPARISON SUMMARY")
    print(f"{'='*60}")
    
    successful = [r for r in results if r['success']]
    
    if successful:
        print(f"\nSuccessful extractions: {len(successful)}/{len(results)}\n")
        
        # Sort by time
        by_time = sorted(successful, key=lambda x: x['time'])
        
        print("Performance (fastest first):")
        for r in by_time:
            print(f"  {r['time']:.2f}s - {r['provider']:12} ({r['model']})")
        
        print(f"\nFastest: {by_time[0]['provider']} ({by_time[0]['time']:.2f}s)")
        print(f"Slowest: {by_time[-1]['provider']} ({by_time[-1]['time']:.2f}s)")
    
    # Show failures
    failures = [r for r in results if not r['success']]
    if failures:
        print(f"\n\nFailed extractions ({len(failures)}):")
        for r in failures:
            print(f"  ✗ {r['provider']} - {r['error'][:50]}...")

In [ ]:
# Compare extraction quality
if successful:
    print("\nData Quality Comparison:")
    print(f"\n{'Provider':<12} {'Invoice#':<15} {'Supplier':<25} {'Amount':<10} {'Items'}")
    print("-" * 80)
    
    for r in successful:
        data = r['data']
        inv_num = str(data.get('invoice_number', 'N/A'))[:14]
        supplier = str(data.get('supplier_name', 'N/A'))[:24]
        amount = str(data.get('total_amount', 'N/A'))[:9]
        items = len(data.get('line_items', [])) if data.get('line_items') else 0
        
        print(f"{r['provider']:<12} {inv_num:<15} {supplier:<25} {amount:<10} {items}")

In [ ]:
# Show full extracted data for comparison
SHOW_FULL_DATA = False  # Set to True to see all extracted data

if SHOW_FULL_DATA and successful:
    for r in successful:
        print(f"\n{'='*60}")
        print(f"{r['provider']} / {r['model']}")
        print(f"{'='*60}")
        print(json.dumps(r['data'], indent=2, ensure_ascii=False))

## 💰 Cost Comparison (Approximate)

Based on typical invoice processing:

| Provider | Model | Cost per invoice | Speed | Quality |
|----------|-------|-----------------|-------|----------|
| Anthropic | Claude Sonnet 4.5 | $0.01-0.03 | Medium | ⭐⭐⭐⭐⭐ |
| OpenAI | GPT-4o | $0.02-0.04 | Medium | ⭐⭐⭐⭐ |
| OpenAI | GPT-4o-mini | $0.001-0.003 | Fast | ⭐⭐⭐ |
| DeepSeek | deepseek-chat | $0.0001-0.0003 | Fast | ⭐⭐⭐⭐ |
| Groq | Llama 3.2 90B | Free* | Very Fast | ⭐⭐⭐ |
| Ollama | Local models | Free | Slow | ⭐⭐ |

*Groq has free tier with rate limits

**Note:** Costs vary based on invoice size and complexity.

## 🎯 Recommendations

### For PDF Invoices
**Use: Anthropic Claude**
- Only provider with native PDF support
- Best quality for scanned documents
- Handles Czech language well

### For Image Invoices
**Budget Option: DeepSeek**
- Very low cost (~100x cheaper than GPT-4)
- Good quality
- Fast processing

**High Quality: OpenAI GPT-4o**
- Excellent quality
- Good with complex layouts
- Reliable

**Fast & Free: Groq**
- Very fast inference
- Free tier available
- Good for simple invoices

**Local/Private: Ollama**
- 100% private (runs locally)
- No API costs
- Requires good hardware
- Lower quality than cloud models

In [ ]:
# Example: Change provider in your config
# 
# Option 1: Edit config/settings.yaml
# ai:
#   provider: "deepseek"
#   model: "deepseek-chat"
#
# Option 2: Set environment variable in .env
# AI_PROVIDER=deepseek
# AI_MODEL=deepseek-chat
# DEEPSEEK_API_KEY=sk-your-key
#
# Option 3: Programmatically
# config.ai.provider = "deepseek"
# config.ai.model = "deepseek-chat"

print("Current configuration:")
print(f"  Provider: {config.ai.provider}")
print(f"  Model: {config.ai.model}")
print(f"\nTo change, edit config/settings.yaml or .env file, then restart kernel.")